In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [26]:
df = pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])

In [27]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [28]:
df.isnull().sum()

,0
text,0
emotion,0


In [29]:
unique_emootion = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emootion:
  emotion_numbers[emotion] = i
  i +=1
df['emotion'] = df['emotion'].map(emotion_numbers)

In [30]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [31]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [32]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [33]:
df['text'] = df['text'].apply(remove_punc)

In [34]:
def remove_numbers(txt):
  new =""
  for i in txt:
    if not i.isdigit():
      new = new +i
  return new
df['text'] = df['text'].apply(remove_numbers)

In [35]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new
df['text'] = df['text'].apply(remove_emojis)

In [36]:
import nltk

In [37]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [38]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [40]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [41]:
def remove(txt):
    words = txt.split()
    cleaned = []

    for i in words:
        if i not in stop_words:
            cleaned.append(i)

    return ' '.join(cleaned)

In [42]:
df['text'] = df['text'].apply(remove)

In [43]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [48]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [52]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [59]:
bow_vectorizer = CountVectorizer()

In [65]:
x_train_bow = bow_vectorizer.fit_transform(x_train)
x_test_bow = bow_vectorizer.transform(x_test)

In [63]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()
nb_model.fit(x_train_bow, y_train)

MultinomialNB()

In [66]:
pred_bow = nb_model.predict(x_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [70]:
tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)

MultinomialNB()

In [71]:
y_pred = nb2_model.predict(x_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


In [72]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(x_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [73]:
log_pred = logistic_model.predict(x_test_tfidf)

In [74]:
print(accuracy_score(y_test,log_pred))

0.8628125
